#Dipendenze

In [ ]:
!pip install --upgrade gradio
!pip install torch torchvision
!pip install deepface

In [2]:
import torch
import gradio as gr
import cv2
import numpy as np
import pickle
import re
import os
from scipy.spatial.distance import cosine
from deepface import DeepFace
import sys

26-02-05 23:31:41 - Directory /root/.deepface has been created
26-02-05 23:31:41 - Directory /root/.deepface/weights has been created


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 1. Caricamento Database e Funzioni di Supporto Generale

**Descrizione:**
In questa fase preliminare viene inizializzato l'ambiente di lavoro. Il sistema tenta di caricare il database biometrico persistente (`.pkl`) contenente i vettori caratteristici (embeddings) degli utenti precedentemente registrati.
Vengono inoltre definite le funzioni *core* di supporto necessarie per l'intero ciclo di vita del software:
* **Preprocessing:** Conversione delle immagini dal formato RGB (standard di Gradio/Web) al formato BGR (standard di OpenCV).
* **Feature Extraction:** Wrapper per la libreria DeepFace che utilizza il modello **ArcFace** per estrarre il vettore numerico rappresentativo del volto.

In [5]:
PATH_DB_OUTPUT = '/content/drive/MyDrive/12/Consegna_Parziale/Gruppo12_embeddings_gallery_consegna.pkl'

try:
    with open(PATH_DB_OUTPUT, 'rb') as f:  # 'rb' sta per Read Binary
        gallery_db = pickle.load(f)

    print("File caricato con successo!")


except FileNotFoundError:
    print(f"Errore: Il file non esiste.")
except Exception as e:
    print(f"Errore durante il caricamento: {e}")

# --- 1. FUNZIONI DI SUPPORTO GENERALE ---

def preprocess_gradio_image(image):
    """Converte da RGB (Gradio) a BGR (OpenCV)"""
    if image is None: return None
    return cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

def get_normalized_embedding(img_bgr, backend='retinaface'):
    """Estrae e normalizza l'embedding (ArcFace)"""
    try:
        embedding_objs = DeepFace.represent(
            img_path = img_bgr,
            model_name = 'ArcFace',
            detector_backend = backend,
            enforce_detection = True,
            align = True
        )
        emb = np.array(embedding_objs[0]["embedding"], dtype=np.float32)
        return emb #/ np.linalg.norm(emb)
    except:
        return None


File caricato con successo!


# 2. Controllo Qualità delle Immagini (Quality Check)

**Descrizione:**
Questa sezione definisce la funzione `check_quality_strict`, un filtro fondamentale per garantire che solo immagini idonee entrino nel database.
Il sistema applica una serie di controlli sequenziali severi secondo le specifiche del progetto:

1.  **Risoluzione e Aspetto:** Verifica che l'immagine sia almeno HD (720x1280) e rispetti rigorosamente il formato **Verticale (Ritratto) 16:9** (con una tolleranza del 5%).
2.  **Illuminazione:** Calcola la luminosità media per scartare immagini sottoesposte (troppo buie) o sovraesposte.
3.  **Nitidezza (Blur Check):** Utilizza la Varianza del Laplaciano per rilevare immagini mosse o sfocate.
4.  **Rilevamento Volto:** Utilizza **RetinaFace** (lo stesso backend usato per la biometria) per assicurarsi che sia presente **uno e un solo volto**. Se il volto non viene trovato o ne vengono trovati più di uno, l'immagine viene scartata.

In [7]:
def check_quality_strict(img_bgr):

    # Check Risoluzione
    h, w, _ = img_bgr.shape
    if w < 720 or h < 1280:
      return False, f"Risoluzione troppo bassa: {w}x{h}. Richiesto almeno HD (720x1280)."

    # 2. CHECK ASPECT RATIO (16:9 Verticale)
    # 9 diviso 16 fa 0.5625.
    ratio = w / h
    target_ratio = 9 / 16
    tolerance = 0.05 # Tolleranza del 5% (accetta da 0.51 a 0.61)

     # Se l'immagine è orizzontale (w > h), avvisiamo l'utente
    if w > h:
         return False, "L'immagine è orizzontale. Per favore scatta la foto in verticale (Ritratto)."

    if not (target_ratio - tolerance < ratio < target_ratio + tolerance):
        return False, f"Formato non valido. L'immagine deve essere in 16:9 Verticale (Rilevato: {ratio:.2f})"

    # Check Luminosità
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    avg_brightness = np.mean(gray)
    if avg_brightness < 20: return False, "Immagine troppo BUIA."
    if avg_brightness > 240: return False, "Immagine SOVRAESPOSTA."

    # Check Blur
    scale = 500 / w
    small_dim = (500, int(h * scale))
    img_small = cv2.resize(img_bgr, small_dim)
    gray_small = cv2.cvtColor(img_small, cv2.COLOR_BGR2GRAY)

    blur_score = cv2.Laplacian(gray_small, cv2.CV_64F).var()

    # Soglia empirica per volti:
    # < 20: Molto sfocato (mosso)
    # 20-40: Un po' soft (tipico selfie con filtro bellezza) -> ACCETTABILE
    # > 40: Nitido
    THRESHOLD_BLUR = 20.0

    if blur_score < THRESHOLD_BLUR:
        return False, f"Immagine SFOCATA o MOSSA (Score: {blur_score:.1f} < {THRESHOLD_BLUR}). Tieni ferma la mano."

    # Check Numero Volti (Max 1)
    try:
        # Usa opencv backend per conteggio veloce
        faces = DeepFace.extract_faces(
            img_path=img_bgr,
            detector_backend='retinaface',
            enforce_detection=True)

        num_faces = len(faces)

        if num_faces > 1:
          return False, f"Troppi volti ({len(faces)}). Massimo consentito: 1."

    except ValueError:
        # CATTURIAMO L'ERRORE SPECIFICO DI DEEPFACE
        # Questo scatta quando RetinaFace guarda la foto e dice "Qui non c'è nessuno".
        return False, "Nessun volto rilevato (Il Detector ha fallito)."

    except Exception as e:
        # CATTURIAMO QUALSIASI ALTRO ERRORE IMPREVISTO
        # Es. Problemi di memoria GPU, librerie corrotte, ecc.
        return False, f"Errore Tecnico durante la detection: {str(e)}"


    return True, "Qualità OK"

# 3. Architettura e Logica del Sistema

---

**Descrizione Generale:**
In questa sezione viene definito il "cervello" dell'applicazione.
Qui implementiamo le classi e le funzioni che gestiscono il flusso operativo delle tre modalità principali del sistema biometrico. Ogni modulo integra i controlli di qualità e i protocolli di sicurezza (Anti-Spoofing) definiti in precedenza.

La logica è suddivisa nei seguenti sottomoduli:

1.  **Logica di Enrollment (Registrazione):** Gestisce l'acquisizione di nuovi utenti, estrae i feature vector e aggiorna il database persistente, con controlli per untenti già registrati.
2.  **Logica di Identificazione (1:N):** Confronta un volto sconosciuto con l'intero database per trovarne l'identità (Open Set). Include la gestione intelligente dei tentativi (Max 3).
3.  **Logica di Verifica (1:1):** Confronta il volto acquisito specificamente con i vettori dell'identità dichiarata. Include la gestione intelligente dei tentativi (Max 3).

In [8]:
# --- CONFIGURAZIONI ---
THRESHOLD_VERIFICATION = 0.458
THRESHOLD_OPENSET = 0.458
SOGLIA_LIVENESS_TF = 0.60

### 3.1 Logica di Enrollment (Registrazione Utenti)

**Descrizione:**
Questo modulo gestisce l'inserimento di nuovi utenti nel sistema. La funzione `logic_enrollment_advanced` implementa un flusso di lavoro robusto composto da:

1.  **Generazione ID Automatico:** La funzione `get_next_student_id` scansiona il database esistente per trovare l'ultimo ID assegnato (es. `subject_05`) e genera automaticamente il successivo (`subject_06`), garantendo univocità e ordine.
2.  **Gestione Multi-Posa (Occhiali):** Il sistema permette di caricare una seconda foto opzionale se l'utente dichiara di portare gli occhiali, migliorando la robustezza del riconoscimento futuro.
3.  **Deduplicazione (Sicurezza):** Prima di salvare, il sistema confronta i vettori del nuovo utente con **tutti** i vettori già presenti nel database. Se la distanza coseno è inferiore alla soglia di sicurezza (`0.40`), la registrazione viene bloccata per impedire che una persona già registrata ottenga un nuovo ID.
4.  **Persistenza:** Se tutti i controlli passano, i vettori vengono salvati nel dizionario in memoria e scritti immediatamente sul file `.pkl`.

In [9]:
def get_next_student_id(database):
    """Genera automaticamente il prossimo ID (es. subject_20)"""
    if not database: return "subject_00"

    max_id = -1
    pattern = re.compile(r"subject_(\d+)")

    for key in database.keys():
        match = pattern.match(key)
        if match:
            current_num = int(match.group(1))
            if current_num > max_id:
                max_id = current_num

    return f"subject_{max_id + 1:02d}"

In [10]:
def logic_enrollment_advanced(img1, img2, has_glasses):
    """
    Gestisce enrollment con 1 o 2 foto e ID automatico.
    """
    # Genera ID
    new_id = get_next_student_id(gallery_db)
    log_msgs = [f"ID Assegnato: {new_id}"]

    # Preparazione lista immagini da processare
    imgs_to_process = []

    # Foto 1 (Standard)
    if img1 is None: return "Manca la Foto 1 (Viso scoperto)."
    imgs_to_process.append(("Foto 1", preprocess_gradio_image(img1)))

    # Foto 2 (Occhiali - Solo se checkbox attiva)
    if has_glasses:
        if img2 is None: return "Hai indicato 'Porto occhiali' ma manca la Foto 2."
        imgs_to_process.append(("Foto 2", preprocess_gradio_image(img2)))

    valid_embeddings = []

    # Ciclo di elaborazione
    for label, img_bgr in imgs_to_process:
        # A. Quality Check Strict
        is_ok, msg = check_quality_strict(img_bgr)
        if not is_ok:
            return f"ERRORE su {label}:\n{msg}"

        # B. Estrazione Embedding
        emb = get_normalized_embedding(img_bgr)
        if emb is None:
            return f"ERRORE su {label}: Il modello biometrico non ha trovato il volto."

        valid_embeddings.append(emb)
        log_msgs.append(f"{label}: Acquisita correttamente.")

    THRESHOLD_DUPLICATE = 0.40

    if gallery_db: # Controlla solo se il DB non è vuoto
        for new_emb in valid_embeddings:
            for existing_id, existing_vectors in gallery_db.items():
                for db_vec in existing_vectors:
                    # Calcola distanza
                    dist = cosine(new_emb, db_vec)

                    if dist < THRESHOLD_DUPLICATE:
                        # TROVATO DUPLICATO!
                        return (
                            f"⛔ REGISTRAZIONE BLOCCATA:\n"
                            f"Questo utente è GIA' REGISTRATO nel sistema come: {existing_id}\n"
                            f"(Distanza rilevata: {dist:.4f} < {THRESHOLD_DUPLICATE})"
                        )

    # Salvataggio nel DB
    if new_id not in gallery_db:
        gallery_db[new_id] = []

    gallery_db[new_id].extend(valid_embeddings)

    # Salvataggio su disco
    with open(PATH_DB_OUTPUT, 'wb') as f:
        pickle.dump(gallery_db, f)

    return "\n".join(log_msgs) + f"\n\n REGISTRAZIONE COMPLETATA!\nTotale vettori salvati: {len(valid_embeddings)}"

### 3.2 Logica di Identificazione (1:N Open Set)

**Descrizione:**
Questo modulo implementa la logica di riconoscimento facciale in modalità "Open Set".
Il sistema confronta il volto in input (Probe) con l'intero database (Gallery) per trovare l'identità più probabile.

**Caratteristiche Principali:**
* **Approccio Open Set:** Il sistema include una "Reject Option". Se il volto non somiglia a nessuno nel database (tutte le distanze > `THRESHOLD_OPENSET`), il sistema restituisce "SCONOSCIUTO" invece di forzare una corrispondenza errata.
* **Ranking:** Se vengono trovati candidati validi, vengono ordinati per somiglianza (distanza minore = match migliore).
* **Gestione Tentativi (Smart Retry):**
    * **Problemi di Qualità:** L'utente ha a disposizione **3 tentativi** per correggere errori di illuminazione, sfocatura o posa.
    * **Spoofing:** Se il modulo Liveness rileva un attacco (maschera/foto), l'accesso viene **bloccato immediatamente** e il sistema si resetta per sicurezza.

In [11]:
def logic_identification_openset(image, use_liveness, attempt_state):

    """1:N Open Set"""

    MAX_ATTEMPTS = 3
    current_attempt = attempt_state + 1
    is_last_attempt = (current_attempt >= MAX_ATTEMPTS)

    # 0. Check Input Base (Questi non contano come tentativi validi)
    if image is None: return "Carica una foto.", 0
    if not gallery_db: return "Database vuoto.", 0

    img_bgr = preprocess_gradio_image(image)

    warnings = []

    # --- STEP 1: CONTROLLO QUALITÀ ---
    is_qual_ok, qual_msg = check_quality_strict(img_bgr)

    if not is_qual_ok:
      if not is_last_attempt:
        return( f"⛔ TENTATIVO {current_attempt}/{MAX_ATTEMPTS} FALLITO (Qualità).\n"
                f"Problema: {qual_msg}\n"
                f"👉 Scatta una foto migliore."
              ), current_attempt
      else:
         warnings.append(f"Qualità Scarsa: {qual_msg}")


    # --- STEP 2: CONTROLLO LIVENESS
    live_msg_output = ""

    if use_liveness:
        is_real, score, msg_live = check_liveness_silentface(img_bgr, SOGLIA_LIVENESS_TF)
        live_msg_output = f"\n🛡️ Liveness: {msg_live} (Score: {score:.2f})"

        if not is_real:
            return (f"Rilevato attacco: {msg_live}\n"
                    f"👉 Usa un volto reale."), 0
    else:

     live_msg_output=""

    header = ""
    if warnings:
        header = (
            f"⚠️ TENTATIVO {current_attempt}/{MAX_ATTEMPTS} (FINALE - FORZATO).\n"
            f"Problemi ignorati: {', '.join(warnings)}.\n"
            f"--------------------------------------------------\n"
        )


    # --- STEP 3: IDENTIFICAZIONE

    # 2. Embedding
    probe_emb = get_normalized_embedding(img_bgr)
    if probe_emb is None: return "Nessun volto rilevato.", 0

    # 3. Ranking

    candidates = []


    for subj_id, vectors in gallery_db.items():

      # Calcola la distanza con TUTTI i vettori di questo student
      distances = [cosine(probe_emb, db_vec) for db_vec in vectors]

      if not distances:
        continue

      # Prendi SOLO la distanza MINIMA per questo studente
      best_dist_for_subject = min(distances)

      # Se supera la soglia (Ti), è un candidato valido
      if best_dist_for_subject < THRESHOLD_OPENSET:
        candidates.append((subj_id, best_dist_for_subject))

    # Ranking
    candidates.sort(key=lambda x: x[1])


    # 5. Formattazione Output
    if not candidates:
      return header + f"❓ SCONOSCIUTO (Reject Option)\nNessun utente sotto la soglia ({THRESHOLD_OPENSET}).\n{live_msg_output}",0

    # Costruiamo la stringa della classifica
    top_match = candidates[0][0] # Il primo è il vincitore
    result_text = f"✅ IDENTIFICATO: {top_match}\n\n🏆 Classifica Corrispondenze (< {THRESHOLD_OPENSET}):\n"

    for i, (subj, dist) in enumerate(candidates):
      # Aggiungiamo un'icona medaglia per i primi 3
      icon = "🥇" if i==0 else "🥈" if i==1 else "🥉" if i==2 else f"{i+1}."
      result_text += f"{icon} {subj} \t [Dist: {dist:.4f}]\n"

    result_text += live_msg_output
    return header + result_text, 0


### 3.3 Logica di Verifica (1:1)

**Descrizione:**
Questo modulo gestisce la verifica dell'identità (Authentication). A differenza dell'identificazione, qui l'utente dichiara esplicitamente chi è (inserendo il proprio ID, es. `subject_01`) e il sistema deve rispondere alla domanda: "Sei davvero chi dici di essere?".

**Funzionalità Chiave:**
* **Confronto Mirato:** Il vettore del volto acquisito viene confrontato **esclusivamente** con i vettori biometrici associati all'ID dichiarato (`claimed_id`).
* **Sicurezza Anti-Spoofing:** Come per l'identificazione, il rilevamento di un attacco (Spoofing) comporta il rifiuto immediato dell'accesso.
* **Tolleranza Errori Utente:** Viene mantenuta la logica dei **3 tentativi** per problemi non fraudolenti (foto buia, mossa o sfocata).

In [12]:
def logic_verification_with_retry(image, claimed_id, use_liveness, attempt_state):
    """
    1:1 Verification con 3 Tentativi per la qualità.
    Lo Spoofing blocca immediatamente.
    """
    # Inizializzazione Stato
    if attempt_state is None: attempt_state = 0

    MAX_ATTEMPTS = 3
    current_attempt = attempt_state + 1
    is_last_attempt = (current_attempt >= MAX_ATTEMPTS)

    # 0. Controlli Preliminari (Non consumano tentativi)
    if image is None: return "⚠️ Dati mancanti: Carica foto.", attempt_state
    if not claimed_id: return "⚠️ Dati mancanti: Inserisci ID.", attempt_state

    claimed_id = claimed_id.strip() # Rimuove spazi accidentali
    if claimed_id not in gallery_db:
        return f"❌ ID '{claimed_id}' inesistente.", 0 # Reset perché è un errore logico

    img_bgr = preprocess_gradio_image(image)

    warnings = [] # Lista per notificare problemi ignorati all'ultimo tentativo

    # --- STEP 1: CONTROLLO QUALITÀ (Gestione Tentativi) ---
    is_ok, msg = check_quality_strict(img_bgr)

    if not is_ok:
        if not is_last_attempt:
            # ERRORE QUALITÀ -> Incrementa tentativo e stop
            return (
                f"⛔ TENTATIVO {current_attempt}/{MAX_ATTEMPTS} FALLITO (Qualità).\n"
                f"Problema: {msg}\n"
                f"👉 Scatta una foto migliore."
            ), current_attempt
        else:
            # ULTIMO TENTATIVO -> Annotiamo e andiamo avanti lo stesso
            warnings.append(f"Qualità ignorata: {msg}")

    # --- STEP 2: LIVENESS (Blocco Immediato) ---
    live_status = ""
    if use_liveness:
        is_real, score, msg_live = check_liveness_silentface(img_bgr, SOGLIA_LIVENESS_TF)
        live_status = f"\nLiveness: {'✅ Reale' if is_real else '⛔ FAKE'} ({score:.2f})"

        if not is_real:
            # FAKE -> Resetta subito a 0 (Nessun tentativo concesso)
            return f"⛔ SPOOFING DETECTED ({msg_live}){live_status}", 0

    # --- STEP 3: EMBEDDING & MATCHING (Tuo Codice Originale) ---

    # Header per avvisare se abbiamo forzato l'ultimo tentativo
    header = ""
    if warnings:
        header = (
            f"⚠️ TENTATIVO {current_attempt}/{MAX_ATTEMPTS} (FINALE - FORZATO).\n"
            f"Problemi: {', '.join(warnings)}.\n"
            f"--------------------------------------------------\n"
        )

    # 2. Embedding
    probe_emb = get_normalized_embedding(img_bgr)
    if probe_emb is None:
        return header + "❌ Nessun volto rilevato.", 0

    # 3. Matching (Solo su claimed_id - ESATTAMENTE COME IL TUO CODICE)
    min_dist = float('inf')

    # Recuperiamo i vettori solo di quell'utente
    user_vectors = gallery_db[claimed_id]

    for db_emb in user_vectors:
        dist = cosine(probe_emb, db_emb)
        if dist < min_dist: min_dist = dist

    res_text = f"Distanza: {min_dist:.4f}{live_status}"

    # Output Finale (Resettiamo lo stato a 0)
    if min_dist < THRESHOLD_OPENSET: # O THRESHOLD_VERIFICATION se ne hai una diversa
        return header + f"✅ VERIFICATO: Benvenuto {claimed_id}.\n{res_text}", 0
    else:
        return header + f"⛔ NON VERIFICATO.\n{res_text}", 0



# 4. Modulo Anti-Spoofing (Liveness Detection)

---

**Descrizione Generale:**
La sicurezza del sistema biometrico si basa sulla capacità di distinguere un volto reale da un tentativo di falsificazione (Spoofing Attack).
Questo modulo integra il framework **Silent-Face-Anti-Spoofing** (MiniFASNet), che analizza l'immagine per rilevare anomalie tipiche di foto stampate, schermi digitali o maschere.

### 4.1 Inizializzazione Ambiente e Caricamento Pesi

---

**Descrizione:**
In questa sezione viene configurato il modulo di **Liveness Detection** (Anti-Spoofing).
Utilizziamo il framework **Silent-Face-Anti-Spoofing** (basato su MiniFASNet), uno dei modelli più leggeri ed efficienti per rilevare attacchi di presentazione (foto, schermi, maschere).

**Dettagli Tecnici dell'Inizializzazione:**
Il codice seguente esegue tre operazioni critiche:
1.  **Download del Repository:** Clona il codice sorgente ufficiale se non è già presente nell'ambiente.
2.  **Iniezione nel Path:** Aggiunge la cartella del repository alle librerie di sistema (`sys.path`) per rendere importabili i moduli interni.
3.  **Il "Path Context Switch" (Trucco del Percorso):**
    * *Problema:* Il codice originale di Silent-Face utilizza percorsi relativi (es. `./resources`) che falliscono se eseguiti dalla cartella principale di Colab (`/content`).
    * *Soluzione:* Lo script memorizza la posizione attuale, si "sposta" fisicamente dentro la cartella del repository (`os.chdir`) per caricare i modelli, e infine ritorna alla posizione originale. Questo garantisce che il caricamento avvenga senza errori di "File Not Found".

In [13]:
# 1. Clona il repository se non esiste già
if not os.path.exists("/content/Silent-Face-Anti-Spoofing"):
    print("Download del modello Silent-Face-Anti-Spoofing...")
    !git clone https://github.com/minivision-ai/Silent-Face-Anti-Spoofing.git

# 2. Aggiunge la cartella al path di Python
if "/content/Silent-Face-Anti-Spoofing" not in sys.path:
    sys.path.append("/content/Silent-Face-Anti-Spoofing")

#Spoofing import
from src.anti_spoof_predict import AntiSpoofPredict
from src.generate_patches import CropImage
from src.utility import parse_model_name

# 4. Inizializzazione GLOBALE dei modelli con TRUCCO DEL PERCORSO
print("Inizializzazione modelli Liveness...")

DEVICE_ID = 0 if torch.cuda.is_available() else "cpu"
# Percorso assoluto per i modelli .pth (questo ci servirà dopo)
MODEL_DIR = "/content/Silent-Face-Anti-Spoofing/resources/anti_spoof_models"

# --- IL FIX FONDAMENTALE ---
# Salviamo dove siamo ora (/content)
current_dir = os.getcwd()

try:
    # Entriamo nella cartella del repository
    os.chdir("/content/Silent-Face-Anti-Spoofing")

    # Ora che siamo dentro, il codice troverà "./resources/detection_model/..."
    liveness_detector = AntiSpoofPredict(DEVICE_ID)
    image_cropper = CropImage()

    print(f"✅ Setup completato. Device: {DEVICE_ID}")

except Exception as e:
    print(f"❌ Errore durante il caricamento: {e}")

finally:
    # IMPORTANTE: Torniamo alla cartella originale (/content)
    # Se non facciamo questo, poi non trova più il tuo Google Drive!
    os.chdir(current_dir)

Download del modello Silent-Face-Anti-Spoofing...
Cloning into 'Silent-Face-Anti-Spoofing'...
remote: Enumerating objects: 376, done.
remote: Counting objects: 100% (116/116), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 376 (delta 92), reused 85 (delta 85), pack-reused 260 (from 2)
Receiving objects: 100% (376/376), 26.04 MiB | 44.67 MiB/s, done.
Resolving deltas: 100% (166/166), done.
Inizializzazione modelli Liveness...
✅ Setup completato. Device: cpu


### 4.2 Funzione di Controllo Liveness

**Descrizione:**
Questa funzione costituisce il cuore operativo del modulo anti-spoofing.
A differenza di implementazioni semplici, questa funzione adotta una strategia **Ensemble**:
1.  **Face Detection:** Individua il volto nell'immagine originale.
2.  **Multi-Model Inference:** Scansiona la cartella dei modelli e, per ciascun modello presente (es. diverse scale o architetture), esegue una predizione separata.
3.  **Preprocessing Adattivo:** Per ogni modello, l'immagine viene ritagliata (`crop`) e scalata dinamicamente in base ai requisiti specifici estratti dal nome del file (tramite `parse_model_name`).
4.  **Score Averaging:** I punteggi di tutti i modelli vengono sommati e normalizzati.
5.  **Decisione:** Il punteggio finale viene confrontato con la soglia configurabile ($T_f$) per determinare l'esito (Vero o Falso).

In [14]:
def check_liveness_silentface(img_bgr, threshold):
    """
    Wrapper per Silent-Face-Anti-Spoofing con SOGLIA CONFIGURABILE.

    Args:
        img_bgr: Immagine BGR (OpenCV)
        threshold: Valore float (0.0 - 1.0).
                   Il punteggio 'Real' deve superare questo valore per essere accettato.
                   - Default consigliato: 0.5
                   - Più severo: 0.7 - 0.9 (Blocca più fake, rischia di bloccare veri)
                   - Più permissivo: 0.1 - 0.4

    Returns:
        is_real (bool), score (float), message (str)
    """
    if img_bgr is None:
        return False, 0.0, "Immagine nulla"

    # Detection del volto
    image_bbox = liveness_detector.get_bbox(img_bgr)

    # Se non trova facce
    if len(image_bbox) == 0:
        return False, 0.0, "Nessun volto rilevato per Liveness Check"

    prediction = np.zeros((1, 3))

    try:
        # Recupera la lista dei modelli dalla cartella globale
        model_files = os.listdir(MODEL_DIR)
        if not model_files:
             return False, 0.0, "Modelli non trovati (controlla MODEL_DIR)"

        # Loop su tutti i modelli (Ensemble)
        for model_name in model_files:
            h_input, w_input, model_type, scale = parse_model_name(model_name)

            # Parametri Crop
            param = {
                "org_img": img_bgr,
                "bbox": image_bbox,
                "scale": scale,
                "out_w": w_input,
                "out_h": h_input,
                "crop": True,
            }
            if scale is None:
                param["crop"] = False

            # Esegue il Crop
            img = image_cropper.crop(**param)

            # Esegue la Predizione
            # Costruisce il path completo del modello
            model_path = os.path.join(MODEL_DIR, model_name)

            # Somma le predizioni dei vari modelli
            prediction += liveness_detector.predict(img, model_path)

    except Exception as e:
        return False, 0.0, f"Errore SilentFace: {str(e)}"

    # --- CALCOLO DELLO SCORE E APPLICAZIONE SOGLIA ---

    # prediction ha shape (1, 3). Indice 1 è la probabilità "Real".
    # Dato che abbiamo sommato le predizioni di N modelli, dividiamo per N per normalizzare.
    num_models = len(model_files)
    real_score = prediction[0][1] / num_models

    # CONFRONTO CON LA SOGLIA UTENTE ($T_f$)
    is_real = real_score > threshold

    if is_real:
        return True, real_score, f"✅ VERO (Score: {real_score:.4f})"
    else:
        return False, real_score, f"⛔ FAKE (Score: {real_score:.4f} sotto soglia {threshold})"

# 5. Interfaccia Grafica Utente (Dashboard Gradio)

---

**Descrizione:**
In questa sezione finale viene costruita la **GUI (Graphical User Interface)** utilizzando la libreria **Gradio**.
L'interfaccia è suddivisa in tre schede (Tab), corrispondenti alle funzionalità principali del progetto.

**Caratteristiche dell'Interfaccia:**
1.  **Tab Enrollment:** Permette la registrazione guidata. Include un elemento dinamico: se l'utente seleziona "Porto gli occhiali", appare automaticamente un secondo box per caricare la foto senza occhiali.
2.  **Tab Identificazione:** Interfaccia per l'accesso 1:N. Gestisce lo stato della sessione (`gr.State`) per contare i tentativi (Max 3) e resettarli dopo un successo o un blocco.
3.  **Tab Verifica:** Interfaccia per il controllo 1:1. Richiede l'ID dell'utente e gestisce anch'essa la logica dei tentativi e l'anti-spoofing.

**Nota Tecnica:**
Viene utilizzato `gr.State` per mantenere la memoria dei tentativi tra un click e l'altro del pulsante, poiché le funzioni di Gradio sono stateless per natura.

In [15]:
# --- 4. INTERFACCIA GRADIO ---

def toggle_img2(checkbox_val):
    return gr.update(visible=checkbox_val)

with gr.Blocks(title="Sistema Biometrico", theme=gr.themes.Soft()) as app:

    gr.Markdown("# 🎓 Sistema Biometrico Universitario")

    # --- TAB 1: ENROLLMENT (Nuovo) ---
    with gr.Tab("➕ Enrollment"):
        gr.Markdown("### Registrazione Nuovo Studente")
        gr.Markdown("⚠️ **Requisiti:** Foto verticale formato 16:9 (Minimo HD 720x1280), ben illuminata, max 1 persone.")

        with gr.Row():
            with gr.Column():
                # Checkbox Occhiali
                chk_glasses = gr.Checkbox(label="Porto gli occhiali", value=False)

                # Foto 1
                img1 = gr.Image(label="Foto 1: Viso Scoperto (o con occhiali se fissi)", sources=["upload"], type="numpy")

                # Foto 2 (Nascosta di default)
                img2 = gr.Image(label="Foto 2: Senza Occhiali", sources=["upload"], type="numpy", visible=False)

                btn_enr = gr.Button("Registra (ID Automatico)", variant="primary")

            with gr.Column():
                out_enr = gr.Textbox(label="Log Sistema", lines=6)

        # Eventi Tab Enrollment
        chk_glasses.change(toggle_img2, inputs=chk_glasses, outputs=img2)
        btn_enr.click(logic_enrollment_advanced, inputs=[img1, img2, chk_glasses], outputs=out_enr)

    # --- TAB 2: IDENTIFICAZIONE ---
    with gr.Tab("🔍 Identificazione (1:N)"):
        gr.Markdown("Identificazione (1:N) con Max 3 Tentativi")
        gr.Markdown("⚠️ **Requisiti:** Foto verticale formato 16:9 (Minimo HD 720x1280), ben illuminata, max 1 persone.")

        with gr.Row():
            with gr.Column():
                id_img = gr.Image(label="Probe", sources=["webcam", "upload"], type="numpy")
                id_live = gr.Checkbox(label="Abilita Liveness Check", value=True)
                btn_id = gr.Button("Identifica")
            with gr.Column():
                out_id = gr.Textbox(label="Risultato", lines=10, max_lines=20)

        state_attempts = gr.State(value=0)

        btn_id.click(logic_identification_openset, inputs=[id_img, id_live, state_attempts], outputs=[out_id, state_attempts])
# --- TAB 3: VERIFICA ---
    with gr.Tab("🔐 Verifica (1:1)"):
        gr.Markdown("Verifica (1:1) con Max 3 Tentativi")
        gr.Markdown("⚠️ **Requisiti:** Foto 16:9, ben illuminata, max 1 persone.")
        with gr.Row():
            with gr.Column():
                ver_img = gr.Image(label="Probe", sources=["webcam", "upload"], type="numpy")
                ver_claim = gr.Textbox(label="ID Dichiarato (es. subject_00)")
                ver_live = gr.Checkbox(label="Abilita Liveness Check", value=True)
                btn_ver = gr.Button("Verifica", variant="primary")
            with gr.Column():
                out_ver = gr.Textbox(label="Risultato", lines=4)

        # --- MODIFICA FONDAMENTALE ---

        # 1. Creiamo la variabile di stato (memoria tentativi)
        state_ver_attempts = gr.State(value=0)

        # 2. Colleghiamo la NUOVA funzione
        btn_ver.click(
            fn=logic_verification_with_retry,           # Nome della funzione aggiornata
            inputs=[ver_img, ver_claim, ver_live, state_ver_attempts], # Input inclusa la memoria
            outputs=[out_ver, state_ver_attempts]       # Output: Testo + Memoria aggiornata
        )

app.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://414d5f921c9b561641.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://414d5f921c9b561641.gradio.live
